In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sc = spark.sparkContext

print("SparkSession créée avec succès !")

SparkSession créée avec succès !


In [3]:
print("Version Spark :", spark.version)

print("Application :", sc.appName)

print("Master :", sc.master)

print(
    "Shuffle partitions :",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Version Spark : 4.0.0
Application : TP1 - Spark SQL
Master : local[*]
Shuffle partitions : 4


In [5]:
print("Mode ANSI :", spark.conf.get("spark.sql.ansi.enabled"))

Mode ANSI : true


In [6]:
try:
    spark.sql("""
        SELECT CAST('abc' AS INT) AS valeur
    """).show()

except Exception as e:
    print("Une erreur est survenue :", e)

{"ts": "2026-11-16 21:01:27.391", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o36.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 2, position 16) ==\n        SELECT CAST('abc' AS INT) AS valeur\n               ^^^^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToN

Une erreur est survenue : [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 2, position 16) ==
        SELECT CAST('abc' AS INT) AS valeur
               ^^^^^^^^^^^^^^^^^^



In [7]:
spark.sql("""
    SELECT try_cast('abc' AS INT) AS valeur
""").show()

+------+
|valeur|
+------+
|  NULL|
+------+



créer le DataFrame

In [10]:
employes = [
    (1, "Sara", "Informatique", "Casablanca", 14000, 28),
    (2, "Youssef", "Informatique", "Rabat", 16500, 35),
    (3, "Imane", "Finance", "Casablanca", 12000, 41)
]

colonnes = ["id", "nom", "departement", "ville", "salaire", "age"]

In [11]:
df_emp = spark.createDataFrame(employes, colonnes)

df_emp.show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
+---+-------+------------+----------+-------+---+



In [12]:
print("Schéma du DataFrame :")
print(df_emp.printSchema())

Schéma du DataFrame :
root
 |-- id: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: long (nullable = true)
 |-- age: long (nullable = true)

None


Explorer les données avec PySpark

In [14]:
df_emp.columns

['id', 'nom', 'departement', 'ville', 'salaire', 'age']

In [15]:
df_emp.dtypes

[('id', 'bigint'),
 ('nom', 'string'),
 ('departement', 'string'),
 ('ville', 'string'),
 ('salaire', 'bigint'),
 ('age', 'bigint')]

In [17]:
df_emp.count()

3

In [18]:
df_emp.describe().show()

+-------+---+-------+------------+----------+------------------+------------------+
|summary| id|    nom| departement|     ville|           salaire|               age|
+-------+---+-------+------------+----------+------------------+------------------+
|  count|  3|      3|           3|         3|                 3|                 3|
|   mean|2.0|   NULL|        NULL|      NULL|14166.666666666666|34.666666666666664|
| stddev|1.0|   NULL|        NULL|      NULL| 2254.624876411447| 6.506407098647712|
|    min|  1|  Imane|     Finance|Casablanca|             12000|                28|
|    max|  3|Youssef|Informatique|     Rabat|             16500|                41|
+-------+---+-------+------------+----------+------------------+------------------+



Partie 2.3 — Créer un DataFrame avec un schéma explicite

In [20]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

In [21]:
schema_emp = StructType([
    StructField("id", IntegerType(), False),
    StructField("nom", StringType(), True),
    StructField("departement", StringType(), True),
    StructField("ville", StringType(), True),
    StructField("salaire", IntegerType(), True),
    StructField("age", IntegerType(), True)
])

In [22]:
df_emp2 = spark.createDataFrame(employes, schema_emp)

df_emp2.show()

df_emp2.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
+---+-------+------------+----------+-------+---+

root
 |-- id: integer (nullable = false)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



In [23]:
df_emp.printSchema()
df_emp2.printSchema()

root
 |-- id: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: long (nullable = true)
 |-- age: long (nullable = true)

root
 |-- id: integer (nullable = false)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



Charger des fichiers avec PySpark

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .getOrCreate()
)

print("Version Spark :", spark.version)

spark.range(5).show()

Version Spark : 4.0.0
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Version Spark :", spark.version)

Version Spark : 4.0.0


In [7]:
df_people_json = spark.read.json("/home/jovyan/data_tp1/people.json")

df_people_json.show()
df_people_json.printSchema()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [10]:
df_csv_a = (
    spark.read
    .option("header", True)
    .csv("/home/jovyan/data_tp1/employes.csv")
)

df_csv_a.show()
df_csv_a.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

root
 |-- id: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: string (nullable = true)
 |-- age: string (nullable = true)



Deuxième méthode : activer inferSchema

In [11]:
df_csv_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/home/jovyan/data_tp1/employes.csv")
)

df_csv_b.show()
df_csv_b.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

root
 |-- id: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



In [12]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

schema_employes = StructType([
    StructField("id", IntegerType(), True),
    StructField("nom", StringType(), True),
    StructField("departement", StringType(), True),
    StructField("ville", StringType(), True),
    StructField("salaire", IntegerType(), True),
    StructField("age", IntegerType(), True)
])

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Version Spark :", spark.version)

Version Spark : 4.0.0


In [1]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

schema_employes = StructType([
    StructField("id", IntegerType(), True),
    StructField("nom", StringType(), True),
    StructField("departement", StringType(), True),
    StructField("ville", StringType(), True),
    StructField("salaire", IntegerType(), True),
    StructField("age", IntegerType(), True)
])

In [3]:
df_employes_schema = (
    spark.read
    .option("header", True)
    .schema(schema_employes)
    .csv("/home/jovyan/data_tp1/employes.csv")
)

df_employes_schema.show()
df_employes_schema.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

root
 |-- id: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



In [4]:
df_csv_c = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/home/jovyan/data_tp1/employes.csv")
)

df_csv_c.show()
df_csv_c.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

root
 |-- id: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



In [6]:
difference = df_employes_schema.exceptAll(df_csv_c).count()

print("Nombre de différences :", difference)

Nombre de différences : 0


Partie 4 : Operations sur Dataframe


In [7]:
df_employes_schema.select("id", "nom", "departement", "ville", "salaire", "age").show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [8]:
df_employes_schema.select("id", "nom", "salaire").show()

+---+-------+-------+
| id|    nom|salaire|
+---+-------+-------+
|  1|   Sara|  14000|
|  2|Youssef|  16500|
|  3|  Imane|  12000|
|  4|   Omar|   9500|
|  5|  Salma|   8800|
|  6|  Hamza|  11000|
|  7|Khadija|  10200|
|  8|   Anas|  13500|
+---+-------+-------+



In [11]:
df_emp_annual = df_employes_schema.withColumn(
    "salaire_annuel",
    df_employes_schema["salaire"] * 12
)

df_emp_annual.show()

+---+-------+------------+----------+-------+---+--------------+
| id|    nom| departement|     ville|salaire|age|salaire_annuel|
+---+-------+------------+----------+-------+---+--------------+
|  1|   Sara|Informatique|Casablanca|  14000| 28|        168000|
|  2|Youssef|Informatique|     Rabat|  16500| 35|        198000|
|  3|  Imane|     Finance|Casablanca|  12000| 41|        144000|
|  4|   Omar|     Finance|    Tanger|   9500| 26|        114000|
|  5|  Salma|          RH|     Rabat|   8800| 30|        105600|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|        132000|
|  7|Khadija|          RH|Casablanca|  10200| 38|        122400|
|  8|   Anas|     Finance|     Rabat|  13500| 45|        162000|
+---+-------+------------+----------+-------+---+--------------+



In [12]:
df_employes_schema.filter(df_employes_schema["salaire"] > 12000).show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [14]:
df_employes_schema.filter((df_employes_schema["departement"] == "Informatique") & (df_employes_schema["salaire"] > 12000)).show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
+---+-------+------------+----------+-------+---+



In [15]:
from pyspark.sql import functions as F

In [16]:
df_emp_profile = df_employes_schema.withColumn(
    "profil",
    F.when(F.col("age") < 30, "Junior")
        .when(F.col("age") < 40, "Intermédiaire")
        .otherwise("Senior")
)

df_emp_profile.show()

+---+-------+------------+----------+-------+---+-------------+
| id|    nom| departement|     ville|salaire|age|       profil|
+---+-------+------------+----------+-------+---+-------------+
|  1|   Sara|Informatique|Casablanca|  14000| 28|       Junior|
|  2|Youssef|Informatique|     Rabat|  16500| 35|Intermédiaire|
|  3|  Imane|     Finance|Casablanca|  12000| 41|       Senior|
|  4|   Omar|     Finance|    Tanger|   9500| 26|       Junior|
|  5|  Salma|          RH|     Rabat|   8800| 30|Intermédiaire|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|       Junior|
|  7|Khadija|          RH|Casablanca|  10200| 38|Intermédiaire|
|  8|   Anas|     Finance|     Rabat|  13500| 45|       Senior|
+---+-------+------------+----------+-------+---+-------------+



In [17]:
df_emp_pp = df_employes_schema.withColumn(
    "profile",
    F.when(df_employes_schema["age"] < 30, "Junior")
        .when(df_employes_schema["age"] < 40, "Intermédiaire")
        .otherwise("Senior")
)

df_emp_pp.show()

+---+-------+------------+----------+-------+---+-------------+
| id|    nom| departement|     ville|salaire|age|      profile|
+---+-------+------------+----------+-------+---+-------------+
|  1|   Sara|Informatique|Casablanca|  14000| 28|       Junior|
|  2|Youssef|Informatique|     Rabat|  16500| 35|Intermédiaire|
|  3|  Imane|     Finance|Casablanca|  12000| 41|       Senior|
|  4|   Omar|     Finance|    Tanger|   9500| 26|       Junior|
|  5|  Salma|          RH|     Rabat|   8800| 30|Intermédiaire|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|       Junior|
|  7|Khadija|          RH|Casablanca|  10200| 38|Intermédiaire|
|  8|   Anas|     Finance|     Rabat|  13500| 45|       Senior|
+---+-------+------------+----------+-------+---+-------------+



In [19]:
df_employes_schema.groupBy("departement").count().show()

+------------+-----+
| departement|count|
+------------+-----+
|Informatique|    3|
|     Finance|    3|
|          RH|    2|
+------------+-----+



In [20]:
df_stats = (
    df_employes_schema
    .groupBy("departement")
    .agg(
        F.count("*").alias("nombre_employes"),
        F.avg("salaire").alias("salaire_moyen"),
        F.max("salaire").alias("salaire_max")
    )
)

df_stats.show()

+------------+---------------+------------------+-----------+
| departement|nombre_employes|     salaire_moyen|salaire_max|
+------------+---------------+------------------+-----------+
|Informatique|              3|13833.333333333334|      16500|
|     Finance|              3|11666.666666666666|      13500|
|          RH|              2|            9500.0|      10200|
+------------+---------------+------------------+-----------+



In [21]:
df_sl = df_employes_schema.orderBy(F.col("salaire").desc()).limit(3)

df_sl.show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



#L'objectif est d'apprendre à manipuler tes DataFrames avec des requêtes SQL classiques.

In [22]:
df_employes_schema.createOrReplaceTempView("employes")

In [23]:
resultat = spark.sql("""
    SELECT *
    FROM employes
""")

resultat.show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [ ]:
req1 = spark.sql(""" 
SELECT * FROM employes WHERE departement = 'Informatique' AND salaire > 12000
""")

req1.show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
+---+-------+------------+----------+-------+---+



: 

In [28]:
req2 = spark.sql (""" 
    select departement , count(*) as total_emp , AVG(salaire) as salaire_moyenne  from employes group by departement HAVING AVG(salaire) > 10000 order by salaire_moyenne desc
    """
)
req2.show()

+------------+---------+------------------+
| departement|total_emp|   salaire_moyenne|
+------------+---------+------------------+
|Informatique|        3|13833.333333333334|
|     Finance|        3|11666.666666666666|
+------------+---------+------------------+



In [29]:
df_employes_schema.createOrReplaceGlobalTempView("employes_global")

In [32]:
spark.sql("""
    SELECT *
    FROM global_temp.employes_global
""").show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



Dans Spark, une vue temporaire classique est accessible uniquement dans la session Spark où elle a été créée.

Une vue globale temporaire permet de partager une vue entre plusieurs sessions Spark appartenant à la même application.

In [4]:
spark2 = spark.newSession()

In [7]:
spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TP1 - Spark SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

df_employes_schema = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/home/jovyan/data_tp1/employes.csv")
)

df_employes_schema.show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [13]:
df_employes_schema.createOrReplaceGlobalTempView("emp_global")

In [14]:
spark.sql("""
    SELECT *
    FROM global_temp.emp_global
""").show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [15]:
resultat_json = spark.sql("""
    SELECT *
    FROM json.`/home/jovyan/data_tp1/people.json`
    WHERE age > 20
""")

resultat_json.show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [16]:
df_stats = (
    df_employes_schema
    .groupBy("departement")
    .agg(
        F.count("*").alias("nombre_employes"),
        F.round(F.avg("salaire"), 0).alias("salaire_moyen"),
        F.max("salaire").alias("salaire_max")
    )
)

df_stats.show()

+------------+---------------+-------------+-----------+
| departement|nombre_employes|salaire_moyen|salaire_max|
+------------+---------------+-------------+-----------+
|Informatique|              3|      13833.0|      16500|
|     Finance|              3|      11667.0|      13500|
|          RH|              2|       9500.0|      10200|
+------------+---------------+-------------+-----------+



In [18]:
df_employes_schema.createOrReplaceTempView("employes")

In [19]:
df_stats_sql = spark.sql("""
    SELECT
        departement,
        COUNT(*) AS nombre_employes,
        ROUND(AVG(salaire), 0) AS salaire_moyen,
        MAX(salaire) AS salaire_max
    FROM employes
    GROUP BY departement
""")

df_stats_sql.show()

+------------+---------------+-------------+-----------+
| departement|nombre_employes|salaire_moyen|salaire_max|
+------------+---------------+-------------+-----------+
|Informatique|              3|      13833.0|      16500|
|     Finance|              3|      11667.0|      13500|
|          RH|              2|       9500.0|      10200|
+------------+---------------+-------------+-----------+



In [20]:
print("Plan d'exécution du DataFrame :")
df_stats.explain(extended=True)

print("Plan d'exécution de la requête SQL :")
df_stats_sql.explain(extended=True)

Plan d'exécution du DataFrame :
== Parsed Logical Plan ==
'Aggregate ['departement], ['departement, 'count(*) AS nombre_employes#92, 'round('avg('salaire), 0) AS salaire_moyen#93, 'max('salaire) AS salaire_max#94]
+- Relation [id#17,nom#18,departement#19,ville#20,salaire#21,age#22] csv

== Analyzed Logical Plan ==
departement: string, nombre_employes: bigint, salaire_moyen: double, salaire_max: int
Aggregate [departement#19], [departement#19, count(1) AS nombre_employes#92L, round(avg(salaire#21), 0) AS salaire_moyen#93, max(salaire#21) AS salaire_max#94]
+- Relation [id#17,nom#18,departement#19,ville#20,salaire#21,age#22] csv

== Optimized Logical Plan ==
Aggregate [departement#19], [departement#19, count(1) AS nombre_employes#92L, round(avg(salaire#21), 0) AS salaire_moyen#93, max(salaire#21) AS salaire_max#94]
+- Project [departement#19, salaire#21]
   +- Relation [id#17,nom#18,departement#19,ville#20,salaire#21,age#22] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+-